In [1]:
# Install and load necessary packages
if (!require("data.table")) install.packages("data.table")
if (!require("readr")) install.packages("readr")
library(data.table)
library(readr)

print("Step 1: Safely Loading Listings and Fixing Columns...")
# FIX: Added 'neighbourhood_cleansed' to grab the actual boroughs
req_cols <- c('id', 'name', 'host_id', 'host_name', 'neighbourhood_group',
              'neighbourhood_cleansed', 'latitude', 'longitude', 'room_type', 'price',
              'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month',
              'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm', 'license')

header <- names(read_csv('/content/listings.csv', n_max = 0, show_col_types = FALSE))
keep_cols <- intersect(req_cols, header)

listings <- read_csv('/content/listings.csv', col_select = all_of(keep_cols), show_col_types = FALSE)
setDT(listings)

# FIX: Rename 'neighbourhood_cleansed' to 'neighbourhood' so it joins perfectly with the Map file
if("neighbourhood_cleansed" %in% names(listings)) {
  setnames(listings, "neighbourhood_cleansed", "neighbourhood")
}

# FIX: Strip the "$" from the Price column so Tableau reads it as a pure number
if("price" %in% names(listings)) {
  listings[, price := as.numeric(gsub("[\\$,]", "", price))]
}

neighbourhoods <- read_csv('/content/neighbourhoods.csv', show_col_types = FALSE)
setDT(neighbourhoods)

print("Step 2: Processing the massive Calendar file...")
calendar <- read_csv('/content/calendar.csv', col_select = c('listing_id', 'date', 'available', 'price'), show_col_types = FALSE)
setDT(calendar)

calendar[, price := as.numeric(gsub("[\\$,]", "", price))]
calendar[, date := as.IDate(date)]
calendar[, month_year := paste0(format(date, "%Y-%m"), "-01")]
calendar[, is_occupied := fifelse(tolower(as.character(available)) %in% c("f", "false"), 1, 0)]

cal_agg <- calendar[, .(
  avg_monthly_price = mean(price, na.rm = TRUE),
  occupancy_rate = mean(is_occupied, na.rm = TRUE)
), by = .(listing_id, month_year)]

rm(calendar)
gc()

print("Step 3: Processing Reviews...")
reviews <- read_csv('/content/reviews.csv', col_select = c('listing_id', 'date'), show_col_types = FALSE)
setDT(reviews)
reviews[, date := as.IDate(date)]
reviews[, month_year := paste0(format(date, "%Y-%m"), "-01")]
rev_agg <- reviews[, .(monthly_review_count = .N), by = .(listing_id, month_year)]
rm(reviews)
gc()

print("Step 4: Merging Datasets...")
monthly_data <- merge(cal_agg, rev_agg, by = c("listing_id", "month_year"), all.x = TRUE)
monthly_data[is.na(monthly_review_count), monthly_review_count := 0]
master_df <- merge(listings, monthly_data, by.x = "id", by.y = "listing_id", all.y = TRUE)

if("neighbourhood_group" %in% names(neighbourhoods) && "neighbourhood_group" %in% names(master_df)) {
  neighbourhoods[, neighbourhood_group := NULL]
}
master_df <- merge(master_df, neighbourhoods, by = "neighbourhood", all.x = TRUE)

print("Step 5: Exporting Master File...")
fwrite(master_df, 'listingscalendarreviewsneighbourhood.csv')
print("ETL Complete! The lightweight file is ready to download.")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: readr



[1] "Step 1: Safely Loading Listings and Fixing Columns..."
[1] "Step 2: Processing the massive Calendar file..."


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,919795,49.2,1473245,78.7,1473245,78.7
Vcells,8674928,66.2,518826920,3958.4,625005248,4768.5


[1] "Step 3: Processing Reviews..."


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,928754,49.7,1473245,78.7,1473245,78.7
Vcells,10655572,81.3,415061536,3166.7,625005248,4768.5


[1] "Step 4: Merging Datasets..."
[1] "Step 5: Exporting Master File..."
[1] "ETL Complete! The lightweight file is ready to download."
